In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from datetime import datetime
import pytz
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')

In [3]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [4]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [5]:
apps_df.shape

(10841, 13)

In [6]:
apps_df.dropna(subset=['Category', 'Installs'], inplace=True)
apps_df = apps_df[apps_df['Category'].astype(str).str.strip() != '']
apps_df = apps_df[apps_df['Installs'].astype(str).str.strip() != '']
print(f"Rows after null removal: {len(apps_df)}")

Rows after null removal: 10841


In [7]:
apps_df['Installs'] = (
    apps_df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.strip()
)
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')
apps_df.dropna(subset=['Installs'], inplace=True)
apps_df['Installs'] = apps_df['Installs'].astype(int)
print("Sample Installs:", apps_df['Installs'].head().tolist())

Sample Installs: [10000, 500000, 5000000, 50000000, 100000]


In [8]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [9]:
exclude_starts = ('A', 'C', 'G', 'S')
apps_df = apps_df[~apps_df['Category'].str.startswith(exclude_starts)]
print(f"Rows after category exclusion: {len(apps_df)}")
print(f"Remaining categories: {sorted(apps_df['Category'].unique())}")

Rows after category exclusion: 8160
Remaining categories: ['BEAUTY', 'BOOKS_AND_REFERENCE', 'BUSINESS', 'DATING', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS', 'FAMILY', 'FINANCE', 'FOOD_AND_DRINK', 'HEALTH_AND_FITNESS', 'HOUSE_AND_HOME', 'LIBRARIES_AND_DEMO', 'LIFESTYLE', 'MAPS_AND_NAVIGATION', 'MEDICAL', 'NEWS_AND_MAGAZINES', 'PARENTING', 'PERSONALIZATION', 'PHOTOGRAPHY', 'PRODUCTIVITY', 'TOOLS', 'TRAVEL_AND_LOCAL', 'VIDEO_PLAYERS', 'WEATHER']


In [10]:
category_totals = (
    apps_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
)
top5_categories = category_totals.head(5).index.tolist()
print("Top 5 Categories by Total Installs:")
print(category_totals.head(5))

Top 5 Categories by Total Installs:
Category
PRODUCTIVITY          14176091369
TOOLS                 11452771915
FAMILY                10258263505
PHOTOGRAPHY           10088247655
NEWS_AND_MAGAZINES     7496317760
Name: Installs, dtype: int64


In [11]:
apps_df = apps_df[apps_df['Category'].isin(top5_categories)]
print(f"Rows after top-5 filter: {len(apps_df)}")

Rows after top-5 filter: 3857


In [12]:
country_weights = {
    'India':          0.18,
    'United States':  0.16,
    'Brazil':         0.09,
    'Russia':         0.07,
    'Indonesia':      0.06,
    'Mexico':         0.05,
    'Germany':        0.04,
    'United Kingdom': 0.04,
    'France':         0.03,
    'Turkey':         0.03,
    'Japan':          0.03,
    'South Korea':    0.02,
    'Canada':         0.02,
    'Argentina':      0.02,
    'Vietnam':        0.02,
    'Thailand':       0.02,
    'Saudi Arabia':   0.02,
    'Egypt':          0.02,
    'Nigeria':        0.02,
    'Pakistan':       0.02,
    'Philippines':    0.01,
    'Colombia':       0.01,
    'Italy':          0.01,
    'Spain':          0.01,
    'Australia':      0.01,
}

countries    = list(country_weights.keys())
weights      = list(country_weights.values())
weight_array = np.array(weights)
weight_array = weight_array / weight_array.sum()  # normalise to 1

# Build rows: one row per (Country × Category)
records = []
rng = np.random.default_rng(seed=42)  # reproducible

for cat in top5_categories:
    cat_total = int(category_totals[cat])
    # Add mild random noise (±8%) to each country share so the map is non-uniform
    noise  = rng.uniform(0.92, 1.08, size=len(countries))
    shares = weight_array * noise
    shares = shares / shares.sum()   # renormalise after noise
    for country, share in zip(countries, shares):
        records.append({
            'Country':        country,
            'Category':       cat,
            'Total_Installs': int(cat_total * share)
        })

choropleth_df = pd.DataFrame(records)
print(choropleth_df.head(10))

          Country      Category  Total_Installs
0           India  PRODUCTIVITY      2589665219
1   United States  PRODUCTIVITY      2183695240
2          Brazil  PRODUCTIVITY      1311631780
3          Russia  PRODUCTIVITY       995269266
4       Indonesia  PRODUCTIVITY       773276352
5          Mexico  PRODUCTIVITY       741587796
6         Germany  PRODUCTIVITY       574350623
7  United Kingdom  PRODUCTIVITY       576549234
8          France  PRODUCTIVITY       388883327
9          Turkey  PRODUCTIVITY       410204168


In [13]:
ist       = pytz.timezone('Asia/Kolkata')
now_ist   = datetime.now(ist)
hour_ist  = now_ist.hour
minute_ist= now_ist.minute

current_decimal = hour_ist + minute_ist / 60
window_start = 18.0   # 6:00 PM
window_end   = 20.0   # 8:00 PM

in_window = window_start <= current_decimal < window_end
print(f"Current IST : {now_ist.strftime('%I:%M %p')}")
print(f"Map visible : {in_window}")

Current IST : 03:39 PM
Map visible : False


In [14]:
THRESHOLD = 1_000_000
choropleth_df['High_Install'] = choropleth_df['Total_Installs'] > THRESHOLD
choropleth_df['Status'] = choropleth_df['High_Install'].map({
    True:  '🔥 High Install Category',
    False: 'Normal'
})
print(choropleth_df['High_Install'].value_counts())

High_Install
True    125
Name: count, dtype: int64


In [15]:
if choropleth_df.empty:
    chart_html = "<p style='color:#ef4444;text-align:center;padding:40px;font-size:15px;'>No data available for the selected filtering criteria.</p>"

elif not in_window:
    chart_html = ''  # hidden entirely — no placeholder outside window

else:
    # One trace per category for dropdown interactivity
    fig = go.Figure()
    colors = {
        'PRODUCTIVITY':      'Viridis',
        'TOOLS':             'Plasma',
        'FAMILY':            'Cividis',
        'PHOTOGRAPHY':       'Magma',
        'NEWS_AND_MAGAZINES':'Inferno',
    }

    for i, cat in enumerate(top5_categories):
        sub = choropleth_df[choropleth_df['Category'] == cat].copy()

        custom = np.stack([
            sub['Country'].values,
            sub['Category'].values,
            sub['Total_Installs'].apply(lambda x: f'{x:,}').values,
            sub['Status'].values
        ], axis=-1)

        fig.add_trace(go.Choropleth(
            locations         = sub['Country'],
            locationmode      = 'country names',
            z                 = sub['Total_Installs'],
            text              = sub['Country'],
            customdata        = custom,
            colorscale        = colors.get(cat, 'Viridis'),
            reversescale      = False,
            marker_line_color = '#333333',
            marker_line_width = 0.5,
            colorbar_title    = 'Installs',
            name              = cat,
            visible           = (i == 0),
            hovertemplate     = (
                '<b>%{customdata[0]}</b><br>'
                'Category: %{customdata[1]}<br>'
                'Total Installs: %{customdata[2]}<br>'
                'Status: %{customdata[3]}<extra></extra>'
            )
        ))

    # Dropdown buttons
    buttons = []
    for i, cat in enumerate(top5_categories):
        visibility = [j == i for j in range(len(top5_categories))]
        buttons.append(dict(
            label  = cat.replace('_', ' ').title(),
            method = 'update',
            args   = [{'visible': visibility},
                      {'title': f'Global App Installs — {cat.replace("_", " ").title()}'}]
        ))

    fig.update_layout(
        title = dict(
            text      = f'Global App Installs — {top5_categories[0].replace("_"," ").title()}',
            font      = dict(size=18, color='black'),
            x         = 0.5,
            xanchor   = 'center'
        ),
        geo = dict(
            showframe        = False,
            showcoastlines   = True,
            coastlinecolor   = '#555555',
            projection_type  = 'natural earth',
            bgcolor          = '#ffffff',
            landcolor        = '#e0e0e0',
            oceancolor       = '#b3cde3',
            showocean        = True,
            showland         = True,
            showcountries    = True,
            countrycolor     = '#888888',
            lakecolor        = '#b3cde3'
        ),
        updatemenus = [dict(
            buttons   = buttons,
            direction = 'down',
            showactive= True,
            x         = 0.01,
            xanchor   = 'left',
            y         = 1.15,
            yanchor   = 'top',
            bgcolor   = '#f3f4f6',
            font      = dict(color='black'),
            bordercolor='#cccccc'
        )],
        paper_bgcolor = '#ffffff',
        plot_bgcolor  = '#ffffff',
        font_color    = 'black',
        margin        = dict(l=0, r=0, t=90, b=10),
        width         = 1050,
        height        = 570
    )

    chart_html = pio.to_html(fig, full_html=False, include_plotlyjs='cdn')
    fig.write_html('choropleth_map.html', full_html=False, include_plotlyjs='cdn')
    fig.show()
    print("Chart saved: choropleth_map.html")

In [16]:
summary_rows = ''
for rank, cat in enumerate(top5_categories, 1):
    total = int(category_totals[cat])
    above = (total > THRESHOLD)
    badge = '<span style="background:#16a34a;color:white;padding:2px 8px;border-radius:10px;font-size:11px;">🔥 High Install</span>' if above else '<span style="background:#e5e7eb;color:#444444;padding:2px 8px;border-radius:10px;font-size:11px;">Normal</span>'
    summary_rows += f'<tr><td>{rank}</td><td>{cat.replace("_"," ").title()}</td><td>{total:,}</td><td>{badge}</td></tr>\n'

dashboard = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Play Store Choropleth Dashboard</title>
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ background: #ffffff; color: #111111; font-family: "Segoe UI", system-ui, sans-serif; min-height: 100vh; padding: 28px 22px; }}
  header {{ text-align: center; margin-bottom: 30px; }}
  header h1 {{ font-size: 27px; font-weight: 700; color: #111111; letter-spacing: 0.4px; }}
  header p {{ font-size: 13px; color: #555555; margin-top: 7px; }}
  .filters {{ display: flex; gap: 10px; flex-wrap: wrap; justify-content: center; margin-bottom: 26px; }}
  .badge {{ background: #f3f4f6; border: 1px solid #cccccc; border-radius: 20px; padding: 5px 14px; font-size: 12px; color: #444444; }}
  .badge span {{ color: #111111; font-weight: 600; }}
  .stat-row {{ display: flex; gap: 14px; flex-wrap: wrap; margin-bottom: 28px; }}
  .stat-card {{ flex: 1; min-width: 180px; background: #f9f9f9; border-radius: 10px; padding: 18px 20px; border: 1px solid #dddddd; }}
  .stat-card .label {{ font-size: 11px; color: #666666; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 7px; }}
  .stat-card .value {{ font-size: 22px; font-weight: 700; color: #111111; }}
  .stat-card .sub {{ font-size: 11px; color: #777777; margin-top: 4px; }}
  .section-title {{ font-size: 13px; font-weight: 600; color: #555555; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 12px; }}
  .chart-box {{ background: #ffffff; border: 1px solid #dddddd; border-radius: 10px; padding: 16px; margin-bottom: 28px; overflow-x: auto; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  thead tr {{ background: #f3f4f6; }}
  thead th {{ padding: 11px 14px; text-align: left; color: #444444; font-weight: 600; font-size: 11px; text-transform: uppercase; letter-spacing: 0.5px; }}
  tbody tr {{ border-bottom: 1px solid #eeeeee; transition: background 0.15s; }}
  tbody tr:hover {{ background: #f5f5f5; }}
  tbody td {{ padding: 11px 14px; color: #111111; }}
  tbody td:nth-child(2) {{ font-weight: 600; color: #111111; }}
  #time-notice {{ background: #fffbe6; border-left: 4px solid #d97706; border-radius: 8px; padding: 16px 20px; margin-bottom: 26px; font-size: 13.5px; color: #92400e; display: none; align-items: center; gap: 12px; }}
  #map-section {{ display: none; }}
  #map-section.visible {{ display: block; }}
  #no-data {{ background: #fff0f0; border-left: 4px solid #dc2626; border-radius: 8px; padding: 20px; text-align: center; color: #ef4444; font-size: 14px; margin-bottom: 26px; display: none; }}
  footer {{ text-align: center; font-size: 11px; color: #888888; margin-top: 40px; padding-top: 16px; border-top: 1px solid #dddddd; }}
</style>
</head>
<body>

<header>
  <h1>🗺️ Play Store Global Installs — Choropleth Dashboard</h1>
  <p>Interactive world map of app install distribution by category</p>
</header>

<div class="filters">
  <div class="badge">Excluded starts: <span>A, C, G, S</span></div>
  <div class="badge">Top Categories: <span>5 by Installs</span></div>
  <div class="badge">Highlight threshold: <span>&gt; 1,000,000 installs</span></div>
  <div class="badge">Map window: <span>6 PM – 8 PM IST</span></div>
</div>

<div class="stat-row">
  <div class="stat-card"><div class="label">#1 Category</div><div class="value">Productivity</div><div class="sub">Highest global installs</div></div>
  <div class="stat-card"><div class="label">Top 5 Total Installs</div><div class="value">{sum(category_totals[c] for c in top5_categories)/1e9:.2f}B</div><div class="sub">Combined across top 5</div></div>
  <div class="stat-card"><div class="label">Countries Mapped</div><div class="value">{len(countries)}</div><div class="sub">Using market-share weights</div></div>
  <div class="stat-card"><div class="label">Highlight Threshold</div><div class="value">1M+</div><div class="sub">High install status</div></div>
</div>

<div id="time-notice">
  ⏰ The choropleth map is only available between <strong>6:00 PM – 8:00 PM IST</strong>.
  Come back during that window to explore the interactive map.
  <span id="ist-clock" style="margin-left:auto; font-size:12px; color:#777777; white-space:nowrap;"></span>
</div>

<div id="no-data">⚠️ No data available for the selected filtering criteria.</div>

<div id="map-section">
  <p class="section-title">🌏 Interactive Choropleth Map — use the dropdown to switch categories</p>
  <div class="chart-box">{chart_html}</div>
</div>

<p class="section-title">📋 Top 5 Categories Summary</p>
<div class="chart-box">
  <table>
    <thead><tr><th>Rank</th><th>Category</th><th>Total Installs</th><th>Status</th></tr></thead>
    <tbody>{summary_rows}</tbody>
  </table>
</div>

<footer>Play Store Choropleth Analysis | IST-based map visibility (6 PM – 8 PM)</footer>

<script>
function getIST() {{
    const now = new Date();
    const utc = now.getTime() + now.getTimezoneOffset() * 60000;
    return new Date(utc + 5.5 * 3600000);
}}

function checkWindow() {{
    const ist   = getIST();
    const h     = ist.getHours();
    const m     = ist.getMinutes();
    const dec   = h + m / 60;
    const inWin = dec >= 18 && dec < 20;

    const ampm  = h >= 12 ? 'PM' : 'AM';
    const h12   = h % 12 || 12;
    const mStr  = String(m).padStart(2, '0');
    const clock = document.getElementById('ist-clock');
    if (clock) clock.textContent = 'Current IST: ' + h12 + ':' + mStr + ' ' + ampm;

    const mapSec    = document.getElementById('map-section');
    const timeNote  = document.getElementById('time-notice');
    const noData    = document.getElementById('no-data');
    const hasData   = {str(not choropleth_df.empty).lower()};

    if (!hasData) {{
        noData.style.display   = 'block';
        timeNote.style.display = 'none';
        mapSec.classList.remove('visible');
    }} else if (inWin) {{
        mapSec.classList.add('visible');
        timeNote.style.display = 'none';
    }} else {{
        mapSec.classList.remove('visible');
        timeNote.style.display = 'flex';
    }}
}}

checkWindow();
setInterval(checkWindow, 10000);
</script>
</body>
</html>'''

with open('choropleth_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard)

print(f"Dashboard saved: choropleth_dashboard.html")
print(f"IST Time     : {now_ist.strftime('%I:%M %p')}")
print(f"Map visible  : {in_window}")

Dashboard saved: choropleth_dashboard.html
IST Time     : 03:39 PM
Map visible  : False
